In [4]:
from pathlib import Path
from dataclasses import dataclass, asdict

@dataclass
class CFG:
    train_path: Path = Path("../data/train.csv")
    test_path: Path = Path("../data/test.csv")
    sub_path: Path = Path("../data/sample_submission.csv")
    pltpd_path: Path = Path("../data/podcast_dataset.csv")

    num_fold: int = 5
    dev_mode: bool = False

    # Model parameters
    n_iter: int = 10000
    max_depth: int = -1
    num_leaves: int = 1024
    colsample_bytree: float = 0.7
    learning_rate: float = 0.02

    objective: str = 'l2'
    metric: str = 'rmse'
    verbosity: int = -1
    max_bin: int = 1024
    
    random_state: int = 42
    shuffle: bool = True
    encoded_columns_start: int = -91
    log_eval: int = 100
    early_stopping: int = 200
    
cfg = CFG() 
asdict(cfg)

{'train_path': PosixPath('../data/train.csv'),
 'test_path': PosixPath('../data/test.csv'),
 'sub_path': PosixPath('../data/sample_submission.csv'),
 'pltpd_path': PosixPath('../data/podcast_dataset.csv'),
 'num_fold': 5,
 'dev_mode': False,
 'n_iter': 10000,
 'max_depth': -1,
 'num_leaves': 1024,
 'colsample_bytree': 0.7,
 'learning_rate': 0.02,
 'objective': 'l2',
 'metric': 'rmse',
 'verbosity': -1,
 'max_bin': 1024,
 'random_state': 42,
 'shuffle': True,
 'encoded_columns_start': -91,
 'log_eval': 100,
 'early_stopping': 200}

In [5]:
from IPython.display import display
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

re_dict = {}
re_dict['podc_dict'] = {
    'Mystery Matters': 0, 'Joke Junction': 1, 'Study Sessions': 2, 'Digital Digest': 3, 
    'Mind & Body': 4, 'Fitness First': 5, 'Criminal Minds': 6, 'News Roundup': 7, 
    'Daily Digest': 8, 'Music Matters': 9, 'Sports Central': 10, 'Melody Mix': 11, 
    'Game Day': 12, 'Gadget Geek': 13, 'Global News': 14, 'Tech Talks': 15, 
    'Sport Spot': 16, 'Funny Folks': 17, 'Sports Weekly': 18, 'Business Briefs': 19, 
    'Tech Trends': 20, 'Innovators': 21, 'Health Hour': 22, 'Comedy Corner': 23, 
    'Sound Waves': 24, 'Brain Boost': 25, "Athlete's Arena": 26, 'Wellness Wave': 27, 
    'Style Guide': 28, 'World Watch': 29, 'Humor Hub': 30, 'Money Matters': 31, 
    'Healthy Living': 32, 'Home & Living': 33, 'Educational Nuggets': 34, 
    'Market Masters': 35, 'Learning Lab': 36, 'Lifestyle Lounge': 37, 
    'Crime Chronicles': 38, 'Detective Diaries': 39, 'Life Lessons': 40, 
    'Current Affairs': 41, 'Finance Focus': 42, 'Laugh Line': 43, 
    'True Crime Stories': 44, 'Business Insights': 45, 'Fashion Forward': 46, 'Tune Time': 47
}
re_dict['genr_dict'] = {'True Crime': 0, 'Comedy': 1, 'Education': 2, 'Technology': 3, 'Health': 4, 'News': 5, 'Music': 6, 'Sports': 7, 'Business': 8, 'Lifestyle': 9}
re_dict['week_dict'] = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
re_dict['time_dict'] = {'Morning': 10, 'Afternoon': 14, 'Evening': 17, 'Night': 21}
re_dict['sent_dict'] = {'Negative': 0, 'Neutral': 1, 'Positive': 2}


def preprocess_df(df):
    df['Episode_Num'] = df['Episode_Title'].str[8:].astype(int)  # Convert to int before log transform
    df = df.drop(columns=['Episode_Title'])

    # Convert categorical variables
    df['Genre'] = df['Genre'].replace(re_dict["genr_dict"])
    df['Podcast_Name'] = df['Podcast_Name'].replace(re_dict["podc_dict"])
    df['Publication_Day'] = df['Publication_Day'].replace(re_dict["week_dict"])
    df['Publication_Time'] = df['Publication_Time'].replace(re_dict["time_dict"])
    df['Episode_Sentiment'] = df['Episode_Sentiment'].replace(re_dict["sent_dict"])

    df.loc[df['Episode_Length_minutes']>121.0, 'Episode_Length_minutes'] = 121.0

    df['Host_Guest_Diff'] = df['Host_Popularity_percentage'] - df['Guest_Popularity_percentage']
    df['Host_Guest_Ratio'] = (df['Host_Popularity_percentage'] / df['Guest_Popularity_percentage']).replace([float('inf'), -float('inf')], pd.NA)

    if "Listening_Time_minutes" in df.columns:
        df['Listening_Episode_Diff'] = df['Episode_Length_minutes'] - df['Listening_Time_minutes']
        df['Listening_Episode_Ratio'] = (df['Episode_Length_minutes'] / df['Listening_Time_minutes']).replace([float('inf'), -float('inf')], pd.NA)

    return df


df_train = pd.read_csv(cfg.train_path, index_col='id')
df_test = pd.read_csv(cfg.test_path, index_col='id')
df_sub = pd.read_csv(cfg.sub_path, index_col='id')

df_pltpd = pd.read_csv(cfg.pltpd_path)
df_pltpd = df_pltpd.dropna(subset=['Listening_Time_minutes'])
df_pltpd = df_pltpd.reset_index(drop=True)
df_pltpd.index = df_pltpd.index + 1000000

df_train = pd.concat([df_train, df_pltpd], axis=0)
df_train["id"] = df_train.index

# is_dev_mode = False
# # is_dev_mode = True
# if is_dev_mode:
#     df_train = df_train.sample(10000, random_state=42)
#     df_test = df_test[:10]
#     df_sub = df_sub[:10]
    
df_train = preprocess_df(df_train)
df_test = preprocess_df(df_test)

# target_col = "Listening_Time_minutes"
# y_train = df_train[target_col].copy()
# df_train = df_train.drop(columns=[target_col])

display(df_train)
display(df_train.describe())
display(df_train.isna().sum())

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
0,0,NaN,0,74.81,3,21,NaN,0.0,2,31.419980,0,98,NaN,NaN,NaN,NaN
1,1,119.80,1,66.95,5,14,75.95,2.0,0,88.012410,1,26,-9.00,0.881501,31.787590,1.361172
2,2,73.90,2,69.97,1,17,8.97,0.0,0,44.925310,2,16,61.00,7.800446,28.974690,1.644952
3,3,67.17,3,57.22,0,10,78.70,2.0,2,46.278240,3,45,-21.48,0.727065,20.891760,1.451438
4,4,110.51,4,80.07,0,14,58.68,3.0,1,75.610310,4,86,21.39,1.364519,34.899690,1.461573
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1047100,33,24.81,9,66.15,0,17,98.63,1.0,1,20.573795,1047100,17,-32.48,0.670688,4.236205,1.205903
1047101,11,92.15,6,89.61,5,21,25.82,2.0,0,76.198459,1047101,9,63.79,3.470565,15.951541,1.209342
1047102,23,112.27,1,26.33,5,21,55.29,0.0,1,107.602135,1047102,24,-28.96,0.476216,4.667865,1.043381
1047103,19,NaN,8,41.47,2,14,33.58,0.0,1,17.220998,1047103,85,7.89,1.234961,NaN,NaN


,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Listening_Episode_Diff
count,797105.000000,705317.000000,797105.000000,797105.000000,797105.000000,797105.000000,646356.000000,797104.000000,797105.000000,797105.000000,7.971050e+05,797105.000000,646356.000000,705317.000000
mean,23.540988,64.408705,4.554814,59.877839,3.028731,15.663856,52.095246,1.357792,0.998145,45.444668,4.133258e+05,51.378954,7.627507,18.679341
std,13.911304,32.981409,2.962341,22.889880,2.022848,4.027274,28.483819,1.149681,0.815531,27.140915,2.598146e+05,28.131239,36.152160,13.566281
min,0.000000,0.000000,0.000000,1.300000,0.000000,10.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,1.000000,-80.170000,-115.540000
25%,12.000000,35.670000,2.000000,39.450000,1.000000,14.000000,28.100000,0.000000,0.000000,23.184220,1.992760e+05,28.000000,-18.280000,8.130000
50%,23.000000,63.770000,5.000000,60.060000,3.000000,17.000000,53.350000,1.000000,1.000000,43.392270,3.985520e+05,52.000000,6.640000,15.643750
75%,36.000000,94.000000,7.000000,79.560000,5.000000,21.000000,76.490000,2.000000,2.000000,64.814620,5.978280e+05,75.000000,33.000000,26.683090
max,47.000000,121.000000,9.000000,119.460000,6.000000,21.000000,119.910000,103.910000,2.000000,119.970000,1.047104e+06,100.000000,113.550000,103.220440


Podcast_Name                        0
Episode_Length_minutes          91788
Genre                               0
Host_Popularity_percentage          0
Publication_Day                     0
Publication_Time                    0
Guest_Popularity_percentage    150749
Number_of_Ads                       1
Episode_Sentiment                   0
Listening_Time_minutes              0
id                                  0
Episode_Num                         0
Host_Guest_Diff                150749
Host_Guest_Ratio               150752
Listening_Episode_Diff          91788
Listening_Episode_Ratio        100172
dtype: int64

In [ ]:
from sklearn.metrics import mean_squared_error

def calc_rmse(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return rmse

calc_rmse([89.693310], [69.530000])

In [78]:
cols_to_compare = ['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage']

df_dup = df_train.copy()
df_dup = df_dup.dropna(subset=['Guest_Popularity_percentage'])
df_dup = df_dup[df_dup.duplicated(subset=cols_to_compare, keep=False)]
df_dup = df_dup.sort_values(cols_to_compare)
df_dup

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio,Listening_Time_minutes_rounded
263468,0,19.30,9,21.51,1,10,96.10,0.0,0,18.236090,263468,1,-74.59,0.223829,1.063910,1.058341,18.2
1043473,0,19.30,0,21.51,1,10,96.10,3.0,0,18.236093,1043473,1,-74.59,0.223829,1.063907,1.058341,18.2
163092,0,93.78,0,68.03,5,10,17.16,1.0,1,71.796010,163092,1,50.87,3.964452,21.983990,1.306201,71.8
348103,0,96.02,0,68.03,5,10,17.16,1.0,1,71.796010,348103,1,50.87,3.964452,24.223990,1.3374,71.8
216180,0,115.56,0,98.62,4,14,2.48,1.0,0,106.422180,216180,2,96.14,39.766129,9.137820,1.085864,106.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044806,47,66.66,6,38.92,1,14,15.98,1.0,1,47.523106,1044806,98,22.94,2.435544,19.136893,1.402686,47.5
123989,47,102.45,6,42.13,6,17,41.29,0.0,1,89.820570,123989,99,0.84,1.020344,12.629430,1.140607,89.8
548936,47,102.45,6,42.13,0,14,41.29,0.0,0,89.820570,548936,99,0.84,1.020344,12.629430,1.140607,89.8
1035367,47,102.45,6,42.13,6,14,41.29,0.0,1,89.820573,1035367,99,0.84,1.020344,12.629427,1.140607,89.8


In [86]:
cols_to_compare = ['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Publication_Time']

df_dup2 = df_train.copy()
# df_dup = df_dup.dropna(subset=['Guest_Popularity_percentage'])
df_dup2 = df_dup2[df_dup2.duplicated(subset=cols_to_compare, keep=False)]
df_dup2 = df_dup2.sort_values(cols_to_compare)
df_dup2

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio,Listening_Time_minutes_rounded
263468,0,19.30,9,21.51,1,10,96.10,0.0,0,18.236090,263468,1,-74.59,0.223829,1.063910,1.058341,18.2
1043473,0,19.30,0,21.51,1,10,96.10,3.0,0,18.236093,1043473,1,-74.59,0.223829,1.063907,1.058341,18.2
340364,0,62.65,0,54.62,1,21,9.30,0.0,2,49.659340,340364,1,45.32,5.873118,12.990660,1.261596,49.7
394230,0,62.65,0,54.62,6,21,5.21,0.0,2,49.659340,394230,1,49.41,10.483685,12.990660,1.261596,49.7
163092,0,93.78,0,68.03,5,10,17.16,1.0,1,71.796010,163092,1,50.87,3.964452,21.983990,1.306201,71.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413021,47,102.36,6,83.30,2,10,5.61,0.0,2,86.602180,413021,99,77.69,14.848485,15.757820,1.181956,86.6
383735,47,39.88,6,29.88,5,21,55.83,0.0,0,36.783300,383735,100,-25.95,0.535196,3.096700,1.084188,36.8
666706,47,37.33,6,29.88,5,21,90.88,0.0,0,36.783300,666706,100,-61.00,0.328785,0.546700,1.014863,36.8
407735,47,66.40,6,49.32,3,17,18.25,2.0,0,55.282330,407735,100,31.07,2.702466,11.117670,1.201107,55.3


In [87]:
# Get IDs from each dataframe
ids_df_dup = set(df_dup['id'])
ids_df_dup2 = set(df_dup2['id'])

# Find common IDs
common_ids = ids_df_dup.intersection(ids_df_dup2)

# Find IDs unique to each dataframe
unique_to_df_dup = ids_df_dup - common_ids
unique_to_df_dup2 = ids_df_dup2 - common_ids

# Print results
print(f"IDs unique to df_dup: {sorted(list(unique_to_df_dup))}")
print(f"Count: {len(unique_to_df_dup)}")
print("\n")
print(f"IDs unique to df_dup2: {sorted(list(unique_to_df_dup2))}")
print(f"Count: {len(unique_to_df_dup2)}")

# If you want to see the full rows with unique IDs from df_dup:
unique_rows_df_dup = df_dup[df_dup['id'].isin(unique_to_df_dup)]

# If you want to see the full rows with unique IDs from df_dup2:
unique_rows_df_dup2 = df_dup2[df_dup2['id'].isin(unique_to_df_dup2)]

# Display the dataframes with unique IDs if needed
print("\nSample of unique rows in df_dup:")
display(unique_rows_df_dup)

print("\nSample of unique rows in df_dup2:")
display(unique_rows_df_dup2)

IDs unique to df_dup: [428, 451, 924, 1930, 5857, 6576, 6577, 6989, 8618, 10803, 11129, 11211, 11952, 12371, 12795, 13045, 13314, 13589, 14216, 15147, 15681, 16007, 16138, 17010, 17300, 17640, 18126, 18183, 18690, 18786, 19276, 19415, 21288, 22529, 22656, 22929, 23179, 23578, 23593, 26907, 28186, 28298, 28849, 29104, 32472, 32546, 33219, 33345, 33630, 34091, 34161, 35295, 35296, 35554, 36189, 36784, 36941, 37308, 37518, 38482, 39151, 41375, 41426, 41621, 42489, 42523, 44126, 44948, 45210, 47414, 48236, 48416, 48559, 49809, 50440, 51224, 51439, 51592, 51650, 51663, 52559, 52779, 52889, 53323, 54946, 56057, 56555, 57071, 57386, 57488, 57780, 57863, 57880, 58500, 58522, 58760, 59804, 60593, 60628, 61005, 61084, 61333, 61372, 62643, 63056, 63082, 63086, 63371, 64043, 64061, 64117, 64264, 65496, 65576, 66314, 66461, 66649, 67992, 68125, 68507, 68568, 68742, 69593, 70446, 70870, 71186, 71918, 71940, 72062, 72336, 74178, 74282, 74326, 74353, 74492, 75499, 75754, 75787, 76318, 78442, 78808, 79

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio,Listening_Time_minutes_rounded
248908,0,47.58,0,47.93,0,17,92.10,0.0,2,43.33421,248908,7,-44.17,0.520413,4.24579,1.097978,43.3
430944,0,47.58,0,47.93,0,14,92.10,2.0,2,43.33421,430944,7,-44.17,0.520413,4.24579,1.097978,43.3
90511,0,17.29,0,30.19,6,10,94.84,1.0,2,1.55190,90511,12,-64.65,0.318326,15.73810,11.141182,1.6
457941,0,17.17,0,30.19,4,14,94.84,1.0,2,1.55190,457941,12,-64.65,0.318326,15.61810,11.063857,1.6
156726,0,79.42,0,90.32,4,21,64.53,0.0,0,63.84193,156726,17,25.79,1.399659,15.57807,1.24401,63.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17010,47,100.73,6,78.78,1,14,0.83,1.0,2,64.26326,17010,94,77.95,94.915663,36.46674,1.567459,64.3
357620,47,99.02,6,71.78,0,17,62.83,0.0,0,51.72721,357620,97,8.95,1.142448,47.29279,1.914273,51.7
567546,47,99.02,6,71.78,0,21,62.83,0.0,0,51.72721,567546,97,8.95,1.142448,47.29279,1.914273,51.7
472565,47,NaN,6,90.82,0,17,40.61,1.0,2,87.29556,472565,97,50.21,2.236395,NaN,NaN,87.3



Sample of unique rows in df_dup2:


,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio,Listening_Time_minutes_rounded
340364,0,62.65,0,54.62,1,21,9.30,0.0,2,49.65934,340364,1,45.32,5.873118,12.99066,1.261596,49.7
394230,0,62.65,0,54.62,6,21,5.21,0.0,2,49.65934,394230,1,49.41,10.483685,12.99066,1.261596,49.7
148438,0,78.76,0,40.72,5,14,27.72,1.0,2,75.28559,148438,2,13.00,1.468975,3.47441,1.04615,75.3
655038,0,88.75,0,40.72,5,14,42.43,1.0,2,75.28559,655038,2,-1.71,0.959698,13.46441,1.178844,75.3
272897,0,49.13,0,48.84,5,14,45.95,0.0,0,30.41250,272897,3,2.89,1.062894,18.71750,1.615454,30.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413021,47,102.36,6,83.30,2,10,5.61,0.0,2,86.60218,413021,99,77.69,14.848485,15.75782,1.181956,86.6
383735,47,39.88,6,29.88,5,21,55.83,0.0,0,36.78330,383735,100,-25.95,0.535196,3.09670,1.084188,36.8
666706,47,37.33,6,29.88,5,21,90.88,0.0,0,36.78330,666706,100,-61.00,0.328785,0.54670,1.014863,36.8
407735,47,66.40,6,49.32,3,17,18.25,2.0,0,55.28233,407735,100,31.07,2.702466,11.11767,1.201107,55.3


In [88]:
df_dup[df_dup.duplicated(subset=cols_to_compare)][["Podcast_Name", "Episode_Num", "Host_Popularity_percentage", "Guest_Popularity_percentage", "Listening_Time_minutes", "Episode_Length_minutes", "Publication_Day", "Publication_Time", "Episode_Sentiment"]]

,Podcast_Name,Episode_Num,Host_Popularity_percentage,Guest_Popularity_percentage,Listening_Time_minutes,Episode_Length_minutes,Publication_Day,Publication_Time,Episode_Sentiment
1043473,0,1,21.51,96.10,18.236093,19.30,1,10,0
348103,0,1,68.03,17.16,71.796010,96.02,5,10,1
468002,0,2,98.62,2.48,106.422180,115.56,5,14,0
1012076,0,2,98.62,2.48,106.422183,115.56,0,14,0
234593,0,5,78.05,80.69,79.181990,93.72,3,17,1
...,...,...,...,...,...,...,...,...,...
1031698,47,97,27.00,46.17,65.727273,68.23,2,21,1
394247,47,97,38.72,94.57,42.043470,58.76,6,10,2
1044806,47,98,38.92,15.98,47.523106,66.66,1,14,1
1035367,47,99,42.13,41.29,89.820573,102.45,6,14,1


In [89]:
ltm_f = df_dup.drop_duplicates(subset=cols_to_compare, keep='first')["Listening_Time_minutes"]
ltm_l = df_dup.drop_duplicates(subset=cols_to_compare, keep='last')["Listening_Time_minutes"]
calc_rmse(ltm_f, ltm_l)

0.803336045366901

In [90]:
ltm_f = df_dup2.drop_duplicates(subset=cols_to_compare, keep='first')["Listening_Time_minutes"]
ltm_l = df_dup2.drop_duplicates(subset=cols_to_compare, keep='last')["Listening_Time_minutes"]
calc_rmse(ltm_f, ltm_l)

13.254635695132372

In [60]:
x = 110
df_dup.iloc[x*30 : (x+1)*30][["Podcast_Name", "Episode_Num", "Host_Popularity_percentage", "Guest_Popularity_percentage", "Listening_Time_minutes", "Episode_Length_minutes", "Publication_Day", "Publication_Time", "Episode_Sentiment"]]

,Podcast_Name,Episode_Num,Host_Popularity_percentage,Guest_Popularity_percentage,Listening_Time_minutes,Episode_Length_minutes,Publication_Day,Publication_Time,Episode_Sentiment
556420,1,99,86.62,44.25,4.015990,18.21,1,10,1
601196,1,99,86.62,82.86,31.290830,53.74,5,14,0
321528,1,100,26.19,93.14,3.876570,11.95,2,14,1
340607,1,100,26.19,72.11,3.876570,11.95,2,14,1
68996,1,100,48.49,83.01,7.274330,38.74,3,17,0
1032919,1,100,48.49,83.01,7.274332,10.72,3,17,0
555783,1,100,70.24,85.69,44.831720,94.04,6,21,1
565232,1,100,70.24,24.39,83.607390,104.13,2,10,1
176510,1,100,87.16,32.72,17.570580,NaN,2,10,0
218635,1,100,87.16,89.71,17.570890,32.05,4,10,0


In [61]:
grouped = df_train.groupby(cols_to_compare)
result = grouped.filter(lambda x: x['Listening_Time_minutes'].nunique() > 1)
result = result.sort_values(cols_to_compare)
result

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio,Listening_Time_minutes_rounded
263468,0,19.30,9,21.51,1,10,96.10,0.0,0,18.236090,263468,1,-74.59,0.223829,1.063910,1.058341,18.2
1043473,0,19.30,0,21.51,1,10,96.10,3.0,0,18.236093,1043473,1,-74.59,0.223829,1.063907,1.058341,18.2
2221,0,55.10,0,68.79,6,14,6.29,1.0,2,35.762540,2221,1,62.50,10.936407,19.337460,1.540718,35.8
344896,0,107.40,0,68.79,5,17,17.40,1.0,1,71.796010,344896,1,51.39,3.953448,35.603990,1.495905,71.8
410859,0,NaN,0,68.42,6,17,72.02,0.0,2,60.835260,410859,2,-3.60,0.950014,NaN,NaN,60.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61038,47,69.20,6,29.32,0,14,84.84,2.0,0,63.532210,61038,100,-55.52,0.345592,5.667790,1.089211,63.5
556562,47,37.33,6,29.32,5,21,35.88,0.0,2,36.783300,556562,100,-6.56,0.817168,0.546700,1.014863,36.8
1034568,47,65.05,6,29.32,5,10,71.65,2.0,2,36.773310,1034568,100,-42.33,0.409211,28.276690,1.768946,36.8
629922,47,26.28,6,76.28,4,17,28.58,3.0,1,6.548230,629922,100,47.70,2.668999,19.731770,4.013298,6.5


In [62]:
x = 110
result.iloc[x*30 : (x+1)*30][["Podcast_Name", "Episode_Num", "Host_Popularity_percentage", "Guest_Popularity_percentage", "Listening_Time_minutes", "Episode_Length_minutes", "Publication_Day", "Publication_Time", "Episode_Sentiment"]]

,Podcast_Name,Episode_Num,Host_Popularity_percentage,Guest_Popularity_percentage,Listening_Time_minutes,Episode_Length_minutes,Publication_Day,Publication_Time,Episode_Sentiment
352993,3,78,23.68,65.11,20.368020,33.99,2,14,0
744644,3,78,23.68,62.24,14.190000,33.67,6,17,1
236720,3,78,23.77,70.85,61.445510,71.30,2,14,1
607678,3,78,23.77,28.41,45.283840,NaN,5,17,1
59971,3,78,41.96,85.37,62.638460,67.51,6,21,2
1025600,3,78,41.96,85.37,62.638462,84.73,6,21,2
198876,3,78,49.44,92.46,8.690000,22.29,3,10,1
552721,3,78,49.44,81.25,8.780000,NaN,3,10,1
677774,3,78,58.95,95.30,78.380890,NaN,1,21,1
1011502,3,78,58.95,95.30,78.380892,NaN,1,21,1


In [63]:
df_train['Listening_Time_minutes_rounded'] = df_train['Listening_Time_minutes'].round(1)

grouped = df_train.groupby(cols_to_compare)
unique_counts = grouped['Listening_Time_minutes_rounded'].nunique()

groups_with_diff = unique_counts[unique_counts > 1].index
result = df_train[df_train.set_index(cols_to_compare).index.isin(groups_with_diff)]

result = result.sort_values(cols_to_compare)
result

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio,Listening_Time_minutes_rounded
2221,0,55.10,0,68.79,6,14,6.29,1.0,2,35.762540,2221,1,62.50,10.936407,19.337460,1.540718,35.8
344896,0,107.40,0,68.79,5,17,17.40,1.0,1,71.796010,344896,1,51.39,3.953448,35.603990,1.495905,71.8
410859,0,NaN,0,68.42,6,17,72.02,0.0,2,60.835260,410859,2,-3.60,0.950014,NaN,NaN,60.8
538961,0,NaN,9,68.42,0,21,86.42,2.0,2,50.138780,538961,2,-18.00,0.791715,NaN,NaN,50.1
399171,0,107.08,0,78.87,3,21,72.68,1.0,1,85.767510,399171,2,6.19,1.085168,21.312490,1.248491,85.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61038,47,69.20,6,29.32,0,14,84.84,2.0,0,63.532210,61038,100,-55.52,0.345592,5.667790,1.089211,63.5
556562,47,37.33,6,29.32,5,21,35.88,0.0,2,36.783300,556562,100,-6.56,0.817168,0.546700,1.014863,36.8
1034568,47,65.05,6,29.32,5,10,71.65,2.0,2,36.773310,1034568,100,-42.33,0.409211,28.276690,1.768946,36.8
629922,47,26.28,6,76.28,4,17,28.58,3.0,1,6.548230,629922,100,47.70,2.668999,19.731770,4.013298,6.5


In [52]:
x = 0
result.iloc[x*30 : (x+1)*30][["Podcast_Name", "Episode_Num", "Host_Popularity_percentage", "Guest_Popularity_percentage", "Listening_Time_minutes", "Episode_Length_minutes", "Publication_Day", "Publication_Time", "Episode_Sentiment"]]

,Podcast_Name,Episode_Num,Host_Popularity_percentage,Guest_Popularity_percentage,Listening_Time_minutes,Episode_Length_minutes,Publication_Day,Publication_Time,Episode_Sentiment
321773,0,60,42.14,68.24,30.427760,NaN,6,14,1
1013894,0,60,42.14,68.24,29.427768,40.48,6,14,1
374046,2,31,81.82,99.43,102.890000,103.87,6,14,2
497554,2,31,81.82,99.43,105.890000,106.79,3,10,2
1030540,2,31,81.82,99.43,105.890008,106.79,3,10,2
78144,3,28,26.84,47.80,21.990000,23.68,3,17,2
488094,3,28,26.84,47.80,20.990000,21.78,2,10,2
163254,5,80,70.26,92.66,55.372760,67.37,2,14,2
428023,5,80,70.26,92.66,55.372760,71.64,2,14,2
569438,5,80,70.26,92.66,53.372760,71.53,2,14,2


20.163309999999996

In [58]:
# Drop duplicate get first by Podcast_Name and Episode_Num
result_f = result.drop_duplicates(subset=['Podcast_Name', 'Episode_Num'], keep='first')["Listening_Time_minutes"]
result_l = result.drop_duplicates(subset=['Podcast_Name', 'Episode_Num'], keep='last')["Listening_Time_minutes"]
calc_rmse(result_f, result_l)

11.042652053609213

In [54]:
grouped.diff()

,Episode_Length_minutes,Genre,Publication_Day,Publication_Time,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio,Listening_Time_minutes_rounded
321773,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1013894,NaN,0.0,0.0,0.0,0.0,0.0,-0.999992,692121.0,0.0,0.0,NaN,NaN,-1.0
374046,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
497554,2.92,0.0,-3.0,-4.0,0.0,0.0,3.000000,123508.0,0.0,0.0,-0.080000,-0.001025,3.0
1030540,0.00,0.0,0.0,0.0,0.0,0.0,0.000008,532986.0,0.0,0.0,-0.000008,-0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
434552,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
486961,14.23,0.0,0.0,0.0,2.0,1.0,-1.000000,52409.0,0.0,0.0,15.230000,0.574786,-1.0
689947,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
710059,6.87,0.0,-4.0,-7.0,0.0,0.0,23.000000,20112.0,0.0,0.0,-16.130000,-0.814779,23.0
